# S47_03 — Embeddings and Vector Databases

Embedding models convert text to dense vectors where semantic similarity maps to geometric proximity. Vector databases index these embeddings for fast approximate nearest-neighbor (ANN) search at scale.

## Embedding models

In [ ]:
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np

# all-MiniLM-L6-v2 — fast, 384 dims, good general-purpose embeddings
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

sentences = [
    'The cat sat on the mat.',
    'A feline rested on a rug.',
    'Dogs enjoy playing fetch.',
    'Machine learning is a subset of artificial intelligence.',
]

embeddings = model.encode(sentences)  # shape: (4, 384)
print(f'Embeddings shape: {embeddings.shape}')
print(f'Vector for sentence 0: [{embeddings[0, :5].round(3)}...]')

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Semantic similarity
sim_matrix = cosine_similarity(embeddings)

print('Cosine similarity matrix:')
labels = ['cat/mat', 'feline/rug', 'dogs', 'ML/AI']
print(f'{"":12}', '  '.join(f'{l:10}' for l in labels))
for i, row in enumerate(sim_matrix):
    print(f'{labels[i]:12}', '  '.join(f'{v:10.3f}' for v in row))

# cat/mat ↔ feline/rug should be high (paraphrases)
# ML/AI should be far from all

## Embedding model comparison

| Model | Dims | Speed | Quality | Notes |
|-------|------|-------|---------|-------|
| all-MiniLM-L6-v2 | 384 | Fast | Good | Default choice for prototyping |
| all-mpnet-base-v2 | 768 | Medium | Better | Stronger quality |
| BAAI/bge-large-en-v1.5 | 1024 | Slow | Excellent | Top MTEB benchmark |
| text-embedding-3-small | 1536 | API | Good | OpenAI, cost-effective |
| text-embedding-3-large | 3072 | API | Best | OpenAI, highest quality |
| voyage-3 | 1024 | API | Excellent | Anthropic's embedding API |

## FAISS — local vector search

In [ ]:
# pip install faiss-cpu
import faiss
import numpy as np

# Create a corpus
corpus = [
    'Gradient descent minimizes loss by following negative gradients.',
    'Backpropagation computes gradients using the chain rule.',
    'Overfitting occurs when a model memorizes training data.',
    'Cross-entropy loss measures classification error.',
    'Batch normalization stabilizes deep network training.',
    'Dropout randomly zeros activations to prevent overfitting.',
    'The attention mechanism computes weighted sums of values.',
    'ReLU is the most common activation function in deep learning.',
]

# Embed the corpus
corpus_embeddings = model.encode(corpus)
d = corpus_embeddings.shape[1]  # 384

# Build FAISS flat (exact) index
index = faiss.IndexFlatIP(d)   # Inner Product = cosine similarity on normalized vectors

# Normalize for cosine similarity
faiss.normalize_L2(corpus_embeddings)
index.add(corpus_embeddings)

print(f'FAISS index contains {index.ntotal} vectors')

In [ ]:
# Search
query = 'How do neural networks avoid overfitting?'
query_embedding = model.encode([query])
faiss.normalize_L2(query_embedding)

scores, indices = index.search(query_embedding, k=3)  # top-3 results

print(f'Query: "{query}"\n')
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    print(f'#{rank} (score={score:.3f}): {corpus[idx]}')

## ChromaDB — persistent vector database

In [ ]:
# pip install chromadb
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# In-memory client (use chromadb.PersistentClient(path='./chroma_db') for disk)
chroma_client = chromadb.Client()

embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

collection = chroma_client.create_collection(
    name='ml_concepts',
    embedding_function=embedding_fn,
)

# Add documents
collection.add(
    documents=corpus,
    ids=[f'doc_{i}' for i in range(len(corpus))],
    metadatas=[{'topic': 'deep learning'} for _ in corpus],
)

print(f'Collection size: {collection.count()}')

In [ ]:
# Query
results = collection.query(
    query_texts=['How do neural networks avoid overfitting?'],
    n_results=3,
)

for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f'dist={dist:.3f}: {doc}')

## Vector database comparison

| Database | Deployment | Scale | Notes |
|----------|-----------|-------|-------|
| **FAISS** | In-process | Millions | Pure search, no metadata filtering |
| **ChromaDB** | In-process or server | Millions | Easy to use, good for prototyping |
| **Qdrant** | Docker or cloud | Billions | Rich filtering, production-ready |
| **Weaviate** | Docker or cloud | Billions | GraphQL API, built-in reranking |
| **Pinecone** | Managed cloud | Billions | Easiest ops; fully managed |
| **pgvector** | Postgres extension | Millions | Use if you're already on Postgres |

Next: [S47_04_retrieval_strategies.ipynb](./S47_04_retrieval_strategies.ipynb)